In [1]:
from pathlib import Path
import pandas as pd

STATIC_PARAMETERS = {"RecordID", "Age", "Gender", "Height", "ICUType", "Weight"}
STATIC_EXCLUDE = {"RecordID", "Age", "Gender", "Height", "ICUType"}

def hhmm_to_minutes(value: str) -> int:
    hours, minutes = value.split(":")
    return int(hours) * 60 + int(minutes)

def parse_patient_file(file_path: Path):
    df = pd.read_csv(file_path)
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")

    record_rows = df.loc[df["Parameter"] == "RecordID", "Value"].dropna()
    patient_id = int(record_rows.iloc[0]) if not record_rows.empty else int(file_path.stem)

    static_df = df[df["Parameter"].isin(STATIC_PARAMETERS)].copy()
    static_df = static_df.drop_duplicates(subset=["Parameter"], keep="last")
    static_values = static_df.set_index("Parameter")["Value"].to_dict()

    static_record = {
        "RecordID": patient_id,
        "Age": static_values.get("Age"),
        "Gender": static_values.get("Gender"),
        "Height": static_values.get("Height"),
        "Weight": static_values.get("Weight"),
    }

    dynamic_df = df[~df["Parameter"].isin(STATIC_EXCLUDE)].copy()
    dynamic_df["Minutes"] = dynamic_df["Time"].map(hhmm_to_minutes)
    dynamic_df["RecordID"] = patient_id
    dynamic_df = dynamic_df[["RecordID", "Time", "Minutes", "Parameter", "Value"]]

    return static_record, dynamic_df

def load_cohort(folder: str):
    folder_path = Path(folder)
    static_records = []
    dynamic_tables = []

    for file_path in sorted(folder_path.glob("*.txt")):
        static_record, dynamic_df = parse_patient_file(file_path)
        static_records.append(static_record)
        dynamic_tables.append(dynamic_df)

    patients_static = pd.DataFrame(static_records).drop_duplicates(subset=["RecordID"])
    patients_static = patients_static.set_index("RecordID").sort_index()

    if dynamic_tables:
        patient_events = pd.concat(dynamic_tables, ignore_index=True)
        patient_events = patient_events.sort_values(["RecordID", "Minutes", "Parameter"]).reset_index(drop=True)
    else:
        patient_events = pd.DataFrame(columns=["RecordID", "Time", "Minutes", "Parameter", "Value"])

    return patients_static, patient_events

In [2]:
train_data_location = "set-a"
patients_static, patient_events = load_cohort(train_data_location)

print(f"Number of patients: {patients_static.shape[0]}")
print(f"Number of dynamic observations: {patient_events.shape[0]}")

display(patients_static.head())
display(patient_events.head(15))

Number of patients: 4000
Number of dynamic observations: 1737980


,Age,Gender,Height,Weight
RecordID,,,,
132539,54.0,0.0,-1.0,-1.0
132540,76.0,1.0,175.3,81.6
132541,44.0,0.0,-1.0,56.7
132543,68.0,1.0,180.3,84.6
132545,88.0,0.0,-1.0,-1.0


,RecordID,Time,Minutes,Parameter,Value
0,132539,00:00,0,Weight,-1.00
1,132539,00:07,7,GCS,15.00
2,132539,00:07,7,HR,73.00
3,132539,00:07,7,NIDiasABP,65.00
4,132539,00:07,7,NIMAP,92.33
5,132539,00:07,7,NISysABP,147.00
6,132539,00:07,7,RespRate,19.00
7,132539,00:07,7,Temp,35.10
8,132539,00:07,7,Urine,900.00
9,132539,00:37,37,HR,77.00


In [9]:
#find unique parameters in patient_events
unique_parameters = patient_events["Parameter"].unique()
print(f"Unique parameters in patient_events: {unique_parameters}")
print(f"length of unique parameters: {len(unique_parameters)}")

Unique parameters in patient_events: <StringArray>
[     'Weight',         'GCS',          'HR',   'NIDiasABP',       'NIMAP',
    'NISysABP',    'RespRate',        'Temp',       'Urine',         'HCT',
         'BUN',  'Creatinine',     'Glucose',        'HCO3',           'K',
          'Mg',          'Na',   'Platelets',         'WBC',       'PaCO2',
        'PaO2',          'pH',     'DiasABP',        'FiO2',         'MAP',
    'MechVent',      'SysABP',        'SaO2',         'ALP',         'ALT',
         'AST',     'Albumin',   'Bilirubin',     'Lactate', 'Cholesterol',
   'TroponinI',   'TroponinT']
Length: 37, dtype: str
length of unique parameters: 37


In [3]:
# Optional dictionary view for convenient per-patient access
patient_store = {
    patient_id: {
        "static": patients_static.loc[patient_id].to_dict(),
        "dynamic": group.drop(columns=["RecordID"]).reset_index(drop=True),
    }
    for patient_id, group in patient_events.groupby("RecordID")
}

sample_patient_id = next(iter(patient_store))
print(f"Example patient id: {sample_patient_id}")
print("Static fields:", patient_store[sample_patient_id]["static"])
display(patient_store[sample_patient_id]["dynamic"].head())

Example patient id: 132539
Static fields: {'Age': 54.0, 'Gender': 0.0, 'Height': -1.0, 'Weight': -1.0}


,Time,Minutes,Parameter,Value
0,00:00,0,Weight,-1.00
1,00:07,7,GCS,15.00
2,00:07,7,HR,73.00
3,00:07,7,NIDiasABP,65.00
4,00:07,7,NIMAP,92.33
